# Applied Machine Learning Studio
## Task 2: Real Estate Housing Price Prediction (California Housing)

### 1. Executive Summary & Formulation
This project develops an end-to-end regression pipeline predicting median housing values across California census block groups.
We contrast Ordinary Least Squares (OLS) with regularized $L_2$ (Ridge) and $L_1$ (Lasso) linear models.

- **Target Variable**: `median_house_value` (continuous USD)
- **Features**: Demographics, geography (`longitude`, `latitude`), room aggregations, income, and categorical `ocean_proximity`.
- **Engineering**: Domain ratios (`rooms_per_household`, `bedrooms_per_room`, `population_per_household`).
- **Core Guardrail**: Preprocessing fitted exclusively on training splits to prevent data leakage.


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import GridSearchCV

from src.data_utils import fetch_dataset, inspect_dataset, split_data
from src.task2_pipeline import prepare_housing_data, build_housing_pipeline
from src.metrics import compute_regression_metrics, metrics_summary_table
from src.plotting import plot_actual_vs_predicted, plot_residuals
from src.predictions import predict_housing_price

print('Environment initialized successfully.')


### 2. Dataset Acquisition & Integrity Inspection
Acquire dataset with automated schema verification and cache to `data/raw/housing.csv`.


In [ ]:
URL = 'https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv'
LOCAL_PATH = '../data/raw/housing.csv'
COLS = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms',
        'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

df = fetch_dataset(URL, LOCAL_PATH, expected_columns=COLS)
summary = inspect_dataset(df)

print(f"Total Records: {summary['num_rows']}, Features: {summary['num_columns']}")
print(f"Missing Values: { {k: v for k, v in summary['missing_counts'].items() if v > 0} }")
print("Target summary ($ USD):\n", df['median_house_value'].describe())


### 3. Exploratory Data Analysis & Multicollinearity
Examine feature distributions, missingness in `total_bedrooms` (207 records), and target correlation.


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
print("Correlation with median_house_value:\n", corr_matrix['median_house_value'].sort_values(ascending=False))


### 4. Leak-Free Train / Test Partitioning (80:20)
Data is split strictly before any learned imputation or scaling parameters are calculated.


In [ ]:
X_train, X_test, y_train, y_test = prepare_housing_data()
print(f"Training set: {X_train.shape}, Held-out test set: {X_test.shape}")


### 5. Composite Feature Engineering & ColumnTransformer Pipeline
We integrate automated domain ratio generation, median imputation for numeric features, standard scaling, and one-hot encoding for `ocean_proximity`.


In [ ]:
# 1. Baseline Ordinary Least Squares
ols_pipe = build_housing_pipeline(LinearRegression())
ols_pipe.fit(X_train, y_train)

# 2. Ridge Regression with 5-Fold Training CV
ridge_pipe = build_housing_pipeline(Ridge())
param_grid_ridge = {'model__alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}
grid_ridge = GridSearchCV(ridge_pipe, param_grid_ridge, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_ridge.fit(X_train, y_train)
best_ridge = grid_ridge.best_estimator_

# 3. Lasso Regression with 5-Fold Training CV
lasso_pipe = build_housing_pipeline(Lasso(max_iter=3000, random_state=42))
param_grid_lasso = {'model__alpha': [1.0, 10.0, 50.0, 100.0, 200.0]}
grid_lasso = GridSearchCV(lasso_pipe, param_grid_lasso, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_lasso.fit(X_train, y_train)
best_lasso = grid_lasso.best_estimator_

print(f"Tuned Ridge best alpha: {grid_ridge.best_params_['model__alpha']}")
print(f"Tuned Lasso best alpha: {grid_lasso.best_params_['model__alpha']}")


### 6. Held-Out Out-of-Sample Performance Comparison
Evaluate RMSE, MAE, and $R^2$ on the identical held-out test dataset.


In [ ]:
models = {
    'LinearRegression (OLS)': ols_pipe,
    'Ridge (Tuned)': best_ridge,
    'Lasso (Tuned)': best_lasso,
}

comparison_results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    comparison_results[name] = compute_regression_metrics(y_test, y_pred, n_features=X_train.shape[1])

comp_df = metrics_summary_table(comparison_results)
print(comp_df[['RMSE', 'MAE', 'R2']])


### 7. Residual Diagnostics & Error Distributions
Inspect actual vs. predicted values and evaluate homoscedasticity assumptions.


In [ ]:
y_pred_best = best_ridge.predict(X_test)
residuals = y_test.values - y_pred_best

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(y_test.values, y_pred_best, alpha=0.3, color='#0d9488', s=20)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='#e11d48', linestyle='--')
ax1.set_title('Actual vs. Predicted Price')
ax1.set_xlabel('Actual ($ USD)')
ax1.set_ylabel('Predicted ($ USD)')

sns.histplot(residuals, kde=True, color='#0f172a', bins=35, ax=ax2)
ax2.set_title('Residual Error Distribution')
ax2.set_xlabel('Residual ($ USD)')
plt.tight_layout()
plt.show()


### 8. Interactive Record Estimator
Test the production prediction helper on custom geographic housing records.


In [ ]:
sample_record = {
    'longitude': -122.23,
    'latitude': 37.88,
    'housing_median_age': 41.0,
    'total_rooms': 880.0,
    'total_bedrooms': 129.0,
    'population': 322.0,
    'households': 126.0,
    'median_income': 8.3252,
    'ocean_proximity': 'NEAR BAY'
}

prediction = predict_housing_price(best_ridge, sample_record)
print(f"Predicted Median Value: {prediction['formatted_price_usd']}")
print(f"Engineered Ratios: {prediction['engineered_features']}")


### 9. Discussion & Architectural Insights
- **Feature Contribution**: `median_income` exhibits the strongest linear correlation ($r \approx 0.69$) with housing value.
- **Regularization Behavior**: Because feature count ($P=14$ after one-hot encoding) is modest relative to observation count ($N \approx 16,500$), OLS, Ridge, and Lasso exhibit comparable $R^2 \approx 0.635$.
- **Model Limitations**: The linear model struggles to capture localized geographic micro-climates without non-linear interaction terms or tree-based spatial partitioning.
